# 21. Does forgetting behave the same way in every run?

The paired study reports exactly 10.0% for the forgotten task in all six completed
comparisons. This notebook measures what is behind that number, in every run, so the
result can be checked for consistency across seeds.

A **seed** is the number that fixes the random choices in a run. Three seeds were
completed, 0, 1 and 2, and each one forgot task 3 in one branch and task 14 in another.
That makes six combinations in total, and this notebook measures all six in one session.

### Why repeating the accuracy alone is not enough

All six already reported 10.0%, so measuring accuracy again would only reproduce a number
we have. Accuracy cannot tell apart two very different models:

- a model that **guesses**, spreading its answers over the ten classes
- a model that gives the **same answer to every image**

Each task is scored on 500 images, 50 of each of 10 classes. A model that always answers
one class is right 50 times, which is exactly 10.0%. A guessing model would land near 10%
but move around between runs, usually between about 7% and 13%. Getting exactly 10.0% six
times points to the second kind of model, not the first.

### What is measured instead

For every one of the six runs, at the start and then every 20 forgetting steps:

1. **Accuracy**, as before.
2. **Where the 500 images go**: how many land in each of the ten classes. One bar of 500
   means one fixed answer.
3. **Input sensitivity**: how much the model's ten output scores change when it is shown a
   completely different image. If this is near zero, the model is ignoring its input
   entirely. This is a stronger statement than "it picked one class".
4. **Weight size**: one number for how large the generated weights are. The forgetting step
   is expected to shrink this.
5. **The other seven tasks**, measured by the same code at the same moments, so their
   normal movement acts as a check that the measurement itself is working.

### Which forgetting code this runs

It calls `study_training.run_request`, the same function the paired study ran.
That matters: the repository also has `trainer.forget`, which looks equivalent but
is not. `run_request` aims at the generated weights after scaling, and
`trainer.forget` aims at the raw generator output before scaling. They shrink the
model by different amounts and do different collateral damage, so only
`run_request` can explain the numbers in your report.

### What this notebook does not do

It trains no source model, writes nothing into the study, and cannot disturb a resume. It
stops when forgetting ends and does not continue into learning task 15. It also does not
test whether forgotten information can be recovered; that is the separate study in
notebooks 16 and 17.

## 1. Load the fixed code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading this fixed code revision.")
if IN_COLAB:
    REPO = Path("/content/cark-paired-study")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has edits. Use a fresh runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a CARK checkout.")
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    implementation = ["uncle", "pyproject.toml", "requirements.txt"]
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--", *implementation], text=True).strip()
    # A later documentation-only commit may contain the same fixed implementation.
    differences = subprocess.check_output(["git", "-C", str(REPO), "diff", CODE_VERSION, actual, "--", *implementation], text=True).strip()
    if dirty or differences:
        raise RuntimeError("Use a clean checkout of CODE_VERSION, or open in Colab. Local training edits are preserved.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies before running this notebook.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))
from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.study_training import restore_rng, run_request
# The progress panel only draws bars, so it cannot change a result. It
# therefore sits outside the pinned implementation, in scripts/, and this
# checkout is detached to CODE_VERSION, which predates it. Try the checkout
# first, then main, then fall back to ordinary bars rather than failing.
sys.path.insert(0, str(REPO / 'scripts'))
try:
    from dashboard import Dashboard
except ImportError:
    try:
        tools = Path.cwd() / '_tools'
        tools.mkdir(exist_ok=True)
        subprocess.run(['git', '-C', str(REPO), 'fetch', '--quiet', 'origin', 'main'], check=True)
        (tools / 'dashboard.py').write_bytes(subprocess.check_output(
            ['git', '-C', str(REPO), 'show', 'origin/main:scripts/dashboard.py']))
        sys.path.insert(0, str(tools))
        from dashboard import Dashboard
        print('Progress panel fetched from main.')
    except Exception as problem:
        print('Progress panel unavailable, using plain bars:', problem)

        class Dashboard:
            """Just enough of the panel to keep the notebook running."""

            def __init__(self, *args, **kwargs): pass
            def __enter__(self): return self
            def __exit__(self, *args): return False
            def note(self, text): print(text, flush=True)

            def bar(self, total, description, unit='step'):
                from uncle.telemetry import progress_bar
                return progress_bar(total, description)

import torch
print("Fixed implementation revision:", CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)

## 2. Choose which runs to measure

`COMBINATIONS` lists every completed comparison in the study: three seeds, each forgetting
task 3 in one branch and task 14 in another. Leave it as it is to measure all six. Shorten
it to one pair if you only want to test that the notebook runs.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"

# Every completed comparison in the study: three seeds, two forgotten tasks each.
# Shorten this list to test the notebook quickly, for example [(0, "3")].
COMBINATIONS = [(0, "3"), (0, "14"),
                (1, "3"), (1, "14"),
                (2, "3"), (2, "14")]
ORDER = "order_01"       # the only learning order the study completed

# How often to stop and measure during the 100 forgetting steps.
MEASURE_EVERY = 20

# Measuring every task costs about a second per task per stop. Keeping this on
# gives the comparison that matters: the forgotten task against the tasks that
# are supposed to survive. Turn it off only if you are in a hurry.
EVALUATE_ALL_TASKS = True

# Results are saved to R2, the same way notebook 20 saves its summary. A Colab
# runtime is deleted when the session ends and takes the local figure and
# tables with it, so a local-only copy is usually lost. This write goes into a
# separate "diagnostics" folder and never touches the study's own files, so it
# cannot affect a future resume. Set it to False to skip saving.
SAVE_TO_R2 = True

# Hand the GPU back when the notebook finishes, so it stops using compute
# units while the tab sits open. This runs in the last cell, after the
# results have been saved and checked, never before.
RELEASE_GPU_WHEN_DONE = True

for seed, task in COMBINATIONS:
    if seed not in (0, 1, 2) or task not in ("3", "14"):
        raise ValueError(f"The study only completed seeds 0-2 forgetting task 3 or 14: got ({seed}, {task}).")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
print("Device:", DEVICE, "" if DEVICE == "cuda" else "(this will be slow; an A100 finishes in well under an hour)")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Runs to measure:", len(COMBINATIONS))

# Match the numerical settings the original runs used, so the generated
# weights come out identical rather than merely similar.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID`, and `R2_SECRET_ACCESS_KEY` through Colab
Secrets or environment variables. Their values are never printed or saved. This notebook
only reads the study; the one optional write goes to a separate diagnostics folder.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Load the saved study

The two checks below refuse to continue if the code revision or the class split differs
from the saved study. Without them the measurements would not be comparable with the
report. The task images are built once here and shared by all six runs.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
path = store.download(PREFIX + "/study.json", CACHE / "cache")
plan = json.loads(path.read_text())

# Refuse to run against a study built by different code or a different class
# split. Without these two checks the numbers below would not be comparable
# with the ones already in the report.
if plan["code_version"] != CODE_VERSION:
    raise ValueError("This study requires a different fixed code revision. Use its recorded notebook revision.")
from uncle.learning_diagnostic import file_hash
if file_hash(REPO / "uncle/task_partition.json") != plan["partition_sha256"]:
    raise ValueError("Class partition differs from the saved study.")

# cleanup_completed=False keeps this notebook from deleting anything in R2.
storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = tuple(plan["orders"][ORDER])
NEW_TASK = plan["new_tasks"][0]

# Only the eight earlier tasks are needed. Task 15 is never learned here.
# Built once and reused by every run, because all six share this order.
def fetch_tasks(include):
    """Build the task datasets, clearing a broken download if there is one.

    torchvision reuses any file already on disk and no checksum is given
    here, so a half-finished or error-page download is kept and every rerun
    fails the same way. Delete it and try once more before giving up.
    """
    import shutil, zipfile
    archive = Path(DATA_ROOT).parent / "tiny-imagenet-200.zip"
    for attempt in (1, 2):
        try:
            return build_tasks(root=DATA_ROOT,
                               partition_path=REPO / "uncle/task_partition.json",
                               partition_seed=42, include=include, download=True)
        except (zipfile.BadZipFile, EOFError, OSError) as problem:
            if attempt == 2:
                raise
            print("The Tiny ImageNet download is unusable:", problem)
            if archive.exists():
                head = archive.open("rb").read(4)
                print(f"   {archive} is {archive.stat().st_size} bytes, starts with {head!r}")
                if not head.startswith(b"PK"):
                    print("   That is not a zip, so the server sent something else.")
                archive.unlink()
            shutil.rmtree(Path(DATA_ROOT), ignore_errors=True)
            print("   Cleared it. Downloading again.")

tasks = fetch_tasks(ORDER_TASKS)
print("Learning order:", ORDER_TASKS)
print("Validation images per task:", len(tasks[ORDER_TASKS[0]]["test"]))

## 5. The measurement

Two helpers. `inspect` measures one task and, when asked, also reports where the images
went and how much the model's output changes between different images. `measure` records
one row for the forgotten task plus the seven protected ones.

In [ ]:
from torch.func import functional_call
from torch.utils.data import DataLoader

CLASSES = plan["config"]["classes_per_task"]
EVAL_BATCH = plan["config"]["eval_batch_size"]

def inspect(hypernet, target, trainer, task, full=False):
    """Measure one task. With full=True also report how the model uses its input.

    Accuracy alone cannot separate a model that guesses from one that answers the
    same class every time, because both can score 10%. Two extra numbers do:

    counts            how many of the 500 images landed in each class. One bar of
                      500 means a single fixed answer.
    input_sensitivity the largest change in any of the ten output scores between
                      one image and another. Near zero means the model produces
                      the same scores no matter what it is shown, so it is not
                      looking at the image at all.
    """
    weights = hypernet.weights_for(task)
    buffers = {name: value.clone() for name, value in trainer.task_buffers[task].items()}
    loader = DataLoader(tasks[task]["test"], batch_size=EVAL_BATCH)
    picked, correct, total = [], 0, 0
    lowest = highest = None
    for images, labels in loader:
        scores = functional_call(target, (weights, buffers),
                                 (images.to(trainer.device),), strict=True)
        choice = scores.argmax(1)
        picked.append(choice.cpu())
        correct += (choice == labels.to(trainer.device)).sum().item()
        total += labels.numel()
        if full:
            # Track the smallest and largest value each output score ever takes.
            batch_low, batch_high = scores.min(dim=0).values, scores.max(dim=0).values
            lowest = batch_low if lowest is None else torch.minimum(lowest, batch_low)
            highest = batch_high if highest is None else torch.maximum(highest, batch_high)
    result = {"accuracy": 100.0 * correct / total}
    if full:
        counts = torch.bincount(torch.cat(picked), minlength=CLASSES).tolist()
        result["class_counts"] = counts
        result["distinct_classes"] = sum(1 for c in counts if c > 0)
        result["chosen_class"] = max(range(CLASSES), key=lambda i: counts[i])
        result["input_sensitivity"] = (highest - lowest).max().item()
    return result

def measure(hypernet, target, trainer, forget_task, step, rows):
    """Record one row of evidence at this point in a forgetting run."""
    hypernet.eval()
    target.eval()
    row = {"step": step, "weight_size": hypernet.raw_for(forget_task).norm().item()}
    row.update(inspect(hypernet, target, trainer, forget_task, full=True))
    row["other_tasks"] = {t: inspect(hypernet, target, trainer, t)["accuracy"]
                          for t in ORDER_TASKS if t != forget_task} if EVALUATE_ALL_TASKS else {}
    rows.append(row)
    print(f"    step {step:>3} | accuracy {row['accuracy']:5.1f}% | weights {row['weight_size']:8.1f} "
          f"| classes {row['distinct_classes']:>2} | input sensitivity {row['input_sensitivity']:.2e}")
    return row

## 6. Run every combination

For each pair, the model is rebuilt from the saved weights and one forgetting request is
applied. Rebuilding matters: forgetting changes the model, so the second branch of a seed
must start from a clean copy rather than from the damaged one.

Each printed row is one stop during a run. Watch the class count fall to 1 and the input
sensitivity fall toward zero.

In [ ]:
import gc

# One panel for the whole sweep. While it is open, every progress bar in the
# project renders into it: the R2 downloads, the forgetting loop, and the
# per-task evaluations. Nothing below has to know the panel exists.
results = []
loaded_seed, state = None, None

with Dashboard(f"Forgetting sweep: {len(COMBINATIONS)} runs") as board:
    runs = board.bar(len(COMBINATIONS), "Runs measured", unit="run")

    for SEED, FORGET_TASK in COMBINATIONS:
        runs.set_postfix_str(f"seed {SEED}, forget task {FORGET_TASK}")
        print(f"\n=== seed {SEED}, forgetting task {FORGET_TASK} ===")

        # Each seed has one source model, shared by both of its forgetting
        # branches. Download it once, then reuse it for the second branch.
        if SEED != loaded_seed:
            board.note(f"downloading the source model for seed {SEED}")
            source_job = next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{SEED}")
            state = storage.load(source_job)
            if state is None or state["report"]["status"] != "complete":
                raise RuntimeError(f"Source model for seed {SEED} is not finished in R2.")
            loaded_seed = SEED
            print("    source model loaded:", source_job["id"])

        # Rebuild the model from scratch for every run. Forgetting changes the
        # model, so the second branch of a seed must start from a clean copy of
        # the saved weights, exactly as the study runner does.
        board.note(f"rebuilding the model for seed {SEED}")
        branch_job = next(j for j in ALL_JOBS
                          if j["id"] == f"branches/{ORDER}/seed_{SEED}/forget_{FORGET_TASK}/learn_{NEW_TASK}")
        requests = tuple(("learn", t) for t in ORDER_TASKS)
        requests += tuple(tuple(op) for op in branch_job["operations"])
        config = Config(**{**plan["config"], "seed": SEED, "device": DEVICE, "requests": requests})

        target = build_target(config)
        hypernet = HyperNetwork(target, config)
        # progress=True so the forgetting loop's own bar joins the panel above.
        trainer = UnCLe(hypernet, config, target, tasks, progress=True)
        checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
        restore_rng(state["all_rng"])   # the same random state the real run had

        # Every task that was learned stays protected, except the forgotten one.
        protected = [t for t in state["seen"] if t != FORGET_TASK]
        board.note(f"forgetting task {FORGET_TASK}, protecting {len(protected)} others")
        rows, losses = [], []
        measure(hypernet, target, trainer, FORGET_TASK, 0, rows)

        # The study's own request runner, not trainer.forget. The two are not
        # the same operation: run_request aims at the generated weights after
        # scaling, while trainer.forget aims at the raw generator output
        # before scaling. Only this one produced the numbers in your report.
        steps = config.burn_in

        def note(row):
            """Called after every update. Measure at the steps we care about."""
            if "loss" in row:
                losses.append(row["loss"])
            step = row.get("step", 0)
            if step and (step % MEASURE_EVERY == 0 or step == steps) and rows[-1]["step"] != step:
                measure(hypernet, target, trainer, FORGET_TASK, step, rows)

        # forget_interval=steps keeps it from copying the optimiser state at
        # every checkpoint, since nothing here resumes. evaluation_interval is
        # set out of range because we take our own richer measurements above.
        run_request(trainer, "forget", FORGET_TASK, protected, None, note,
                    lambda active: None, forget_interval=steps,
                    evaluation_interval=10 ** 9)

        results.append({"seed": SEED, "forget_task": FORGET_TASK, "order": ORDER,
                        "source_id": f"sources/{ORDER}/seed_{SEED}", "branch_id": branch_job["id"],
                        "burn_in": config.burn_in, "beta": config.beta, "gamma": config.gamma,
                        "final_loss": losses[-1], "rows": rows})
        runs.update(1)

        # Free the GPU before the next run, so six models do not pile up.
        del trainer, hypernet, target
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    runs.close()
    board.note(f"measured {len(results)} runs")

print(f"\nMeasured {len(results)} runs.")

## 7. Compare across seeds

The first table is the forgotten task in every run. The second is the seven tasks that
were supposed to survive. The verdict underneath states whether the result is consistent.

In [ ]:
def summarise(run):
    """Turn one run's rows into the few numbers worth comparing across seeds."""
    rows, first, last = run["rows"], run["rows"][0], run["rows"][-1]
    constant = next((r["step"] for r in rows if r["distinct_classes"] == 1), None)
    others_at_20 = next((r["other_tasks"] for r in rows if r["step"] == 20), {})
    chance = 100.0 / CLASSES
    return {
        "seed": run["seed"], "forget": run["forget_task"],
        "start_accuracy": first["accuracy"], "end_accuracy": last["accuracy"],
        "classes_before": first["distinct_classes"], "classes_after": last["distinct_classes"],
        "chosen_class": last["chosen_class"] if last["distinct_classes"] == 1 else None,
        "constant_from_step": constant,
        "sensitivity_before": first["input_sensitivity"],
        "sensitivity_after": last["input_sensitivity"],
        "weights_before": first["weight_size"], "weights_after": last["weight_size"],
        "weights_left": last["weight_size"] / first["weight_size"],
        "others_at_chance_step20": sum(1 for v in others_at_20.values() if abs(v - chance) < 0.05),
        "others_total": len(others_at_20),
        "others_recovered_by_end": sum(1 for v in last["other_tasks"].values() if v > chance + 0.05),
    }

table = [summarise(run) for run in results]

print("THE FORGOTTEN TASK, IN EVERY RUN")
print("seed forget | start%   end% | classes after | class | constant from | weights left | input sensitivity")
print("-" * 112)
for r in table:
    seat = "-" if r["chosen_class"] is None else str(r["chosen_class"])
    step = "never" if r["constant_from_step"] is None else f"step {r['constant_from_step']}"
    print(f"{r['seed']:>4} {r['forget']:>6} | {r['start_accuracy']:6.1f} {r['end_accuracy']:6.1f} |"
          f" {r['classes_after']:>13} | {seat:>5} | {step:>13} | {r['weights_left']:11.1%} |"
          f" {r['sensitivity_before']:.2e} -> {r['sensitivity_after']:.2e}")

print()
print("THE OTHER SEVEN TASKS, WHICH WERE SUPPOSED TO SURVIVE")
print("seed forget | at chance after 20 steps | above chance at the end")
print("-" * 66)
for r in table:
    print(f"{r['seed']:>4} {r['forget']:>6} | {r['others_at_chance_step20']:>11} of {r['others_total']:<11}"
          f" | {r['others_recovered_by_end']:>11} of {r['others_total']}")

print()
print("VERDICT")
all_constant = all(r["classes_after"] == 1 for r in table)
all_exact = all(abs(r["end_accuracy"] - 100.0 / CLASSES) < 1e-9 for r in table)
seats = sorted({r["chosen_class"] for r in table if r["chosen_class"] is not None})
if all_constant:
    print(f"  All {len(table)} runs end with one fixed answer for every image.")
    print(f"  The class chosen was {seats}, so the collapse is {'always the same class' if len(seats) == 1 else 'not tied to one class'}.")
    print("  This is consistent across seeds, so it is a property of the method, not of one run.")
else:
    spread = [f"seed {r['seed']} forget {r['forget']}" for r in table if r["classes_after"] > 1]
    print("  Not every run collapses to one class. Still spread:", ", ".join(spread))
    print("  The single-answer explanation does not cover every run. Share this output.")
print(f"  Every run ended at exactly {100.0/CLASSES:.1f}%: {all_exact}")
damaged = [r for r in table if r["others_at_chance_step20"] == r["others_total"] and r["others_total"]]
print(f"  Runs where ALL other tasks were also at chance after 20 steps: {len(damaged)} of {len(table)}")

## 8. Chart

Three panels, one line per run. Left: accuracy of the forgotten task. Middle: the size of
the generated weights as a share of where each run started. Right: input sensitivity on a
log scale, where falling toward the bottom means the model has stopped reading the image.

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime, timezone

EXPORT = CACHE / "diagnostics" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPORT.mkdir(parents=True, exist_ok=True)

figure, axes = plt.subplots(1, 3, figsize=(16, 4.6))
colours = plt.cm.viridis([i / max(1, len(results) - 1) for i in range(len(results))])

for run, colour in zip(results, colours):
    steps = [r["step"] for r in run["rows"]]
    label = f"seed {run['seed']}, forget {run['forget_task']}"
    axes[0].plot(steps, [r["accuracy"] for r in run["rows"]],
                 color=colour, marker="o", linewidth=1.8, label=label)
    # Weight size divided by its own starting value, so all six share a scale.
    start = run["rows"][0]["weight_size"]
    axes[1].plot(steps, [r["weight_size"] / start for r in run["rows"]],
                 color=colour, marker="o", linewidth=1.8, label=label)
    axes[2].plot(steps, [max(r["input_sensitivity"], 1e-12) for r in run["rows"]],
                 color=colour, marker="o", linewidth=1.8, label=label)

axes[0].axhline(100.0 / CLASSES, color="black", linestyle=":", linewidth=1)
axes[0].set_ylabel("accuracy (%)"); axes[0].set_title("Forgotten task accuracy")
axes[1].set_ylabel("share of starting size"); axes[1].set_ylim(bottom=0)
axes[1].set_title("Generated weights shrinking")
axes[2].set_yscale("log"); axes[2].set_ylabel("largest change in output score")
axes[2].set_title("Input sensitivity (lower means ignoring the image)")
for axis in axes:
    axis.set_xlabel("forgetting step")
axes[0].legend(fontsize=7)

figure.suptitle(f"{STUDY_ID} | {ORDER} | {len(results)} runs")
figure.tight_layout()
figure.savefig(EXPORT / "seed_consistency.png", dpi=180)
figure.savefig(EXPORT / "seed_consistency.svg")
plt.show()
print("Figure written to", EXPORT)

## 9. Save

Writes a JSON summary, a one-row-per-run comparison table, and a per-step table next to the
figure, then copies them to R2 under a diagnostics folder separate from the study's own
files. Keep this on: a Colab runtime is deleted when it ends, and the local copies go with
it. Set `SAVE_TO_R2 = False` in section 2 to skip the upload.

In [ ]:
import csv

summary = {
    "study_id": STUDY_ID, "code_version": CODE_VERSION, "order": ORDER,
    "device": DEVICE, "classes_per_task": CLASSES,
    "validation_images": sum(results[0]["rows"][0]["class_counts"]),
    "measure_every": MEASURE_EVERY, "evaluated_all_tasks": EVALUATE_ALL_TASKS,
    "combinations": [[s, t] for s, t in COMBINATIONS],
    "comparison": table, "runs": results,
    "reading": ("One distinct class with near-zero input sensitivity means the model "
                "returns the same scores whatever image it is shown. On a balanced set "
                "that is exactly chance by arithmetic, not by guessing. It does not show "
                "that the task's information was removed."),
}
(EXPORT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

with (EXPORT / "comparison.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(table[0]))
    writer.writeheader()
    writer.writerows(table)

with (EXPORT / "steps.csv").open("w", newline="", encoding="utf-8") as stream:
    columns = ["seed", "forget_task", "step", "accuracy", "weight_size",
               "distinct_classes", "chosen_class", "input_sensitivity"] + \
              [f"class_{i}" for i in range(CLASSES)] + [f"task_{t}" for t in ORDER_TASKS]
    writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    for run in results:
        for row in run["rows"]:
            line = {"seed": run["seed"], "forget_task": run["forget_task"],
                    "step": row["step"], "accuracy": row["accuracy"],
                    "weight_size": row["weight_size"],
                    "distinct_classes": row["distinct_classes"],
                    "chosen_class": row["chosen_class"],
                    "input_sensitivity": row["input_sensitivity"]}
            line.update({f"class_{i}": c for i, c in enumerate(row["class_counts"])})
            line.update({f"task_{t}": v for t, v in row["other_tasks"].items()})
            writer.writerow(line)

print("Saved locally:")
for item in sorted(EXPORT.iterdir()):
    print("  ", item)

if SAVE_TO_R2:
    # A separate folder. The study's own files are never touched.
    destination = PREFIX + "/diagnostics/" + EXPORT.name
    # The same panel as the sweep. R2 transfer bars join it on their own.
    with Dashboard("Saving results") as board:
        files = sorted(EXPORT.iterdir())
        sending = board.bar(len(files), "Upload diagnostics")
        for item in files:
            sending.set_postfix_str(item.name)
            store.upload(item, destination + "/" + item.name)
            sending.update(1)
        sending.close()
    print("Uploaded to", destination)
    print("Share this path:", destination)
    SAVED_LOCATION = destination
else:
    SAVED_LOCATION = str(EXPORT)
    print()
    print("SAVE_TO_R2 is off, so the only copy is inside this runtime.")
    print("It will be lost when the runtime ends.")

# The release cell reads this. The GPU is only handed back once every
# expected file is on disk, so a failed save can never cost you the run.
EXPECTED_FILES = ["comparison.csv", "seed_consistency.png",
                  "seed_consistency.svg", "steps.csv", "summary.json"]
SAVE_VERIFIED = sorted(item.name for item in EXPORT.iterdir()) == EXPECTED_FILES
print()
print("All expected files present:", SAVE_VERIFIED)

## 10. Hand the GPU back

Releases the Colab runtime so it stops using compute units once the work is done.

It refuses to release if any expected output file is missing, so a failed save can never
cost you the run. Releasing ends the session: the variables above are discarded, and
rerunning means starting again from section 1. The files already saved to R2 are not
affected.

In [ ]:
if not SAVE_VERIFIED:
    print("Not releasing the GPU: some output files are missing.")
    print("Rerun section 9 before closing this runtime, or the results are lost.")
elif not RELEASE_GPU_WHEN_DONE:
    print("Results saved to:", SAVED_LOCATION)
    print("RELEASE_GPU_WHEN_DONE is off, so the runtime stays connected.")
elif not IN_COLAB:
    print("Results saved to:", SAVED_LOCATION)
    print("Not running in Colab, so there is no runtime to release.")
else:
    print("Results saved to:", SAVED_LOCATION)
    print("Share that path.")
    # Free the GPU memory first, then hand the machine back. Nothing after
    # unassign runs, because it ends the session immediately.
    import gc
    for name in ("results", "state", "tasks"):
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    from google.colab import runtime
    print("Releasing the Colab runtime now.")
    runtime.unassign()

## What to do with the output

Send me the two tables from section 7 and the verdict under them. That is the whole result.

**If every run ends with one class and near-zero input sensitivity**, the finding is
consistent across seeds. It is then a property of the method rather than an accident of
one run, and the report's wording has to change. Replace "random guessing would score
about 10%" with the real reason: the model gives the same answer to every image, and the
test set holds 50 images of each of 10 classes, so one fixed answer scores exactly 10.0%.
Drop the phrase "chance level", because the model is not at chance, it has stopped
responding to its input.

**If some runs still spread their answers** across several classes while scoring 10.0%,
then the single-answer explanation does not cover everything and we look again.

**Watch the second table too.** It counts how many of the seven protected tasks were also
at chance after 20 steps. Seed 0 showed all seven collapsing, which is far more damage
than the report describes. If that repeats across seeds, it is a finding in its own right.

Once this is settled, the next experiment continues into the learning of task 15 to see
when the other tasks recover.